# PCA, SVD, and Eigendecomposition: Olivetti Faces

This notebook reduces 64×64 face images to a smaller number of principal components and visualizes the reconstruction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X, y = faces.data, faces.target

n_components = 100
pca = PCA(n_components=n_components, svd_solver='randomized', random_state=42)
X_projected = pca.fit_transform(X)
X_reconstructed = pca.inverse_transform(X_projected)

print(f'Original shape: {X.shape}')
print(f'Compressed shape: {X_projected.shape}')
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.2%}')

plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of principal components')
plt.ylabel('Cumulative explained variance')
plt.ylim(0, 1.05)
plt.grid()
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_reconstructed[i].reshape(faces.images.shape[1:]), cmap='gray')
    ax.set_title(f'Reconstruction {i}')
    ax.axis('off')
plt.tight_layout()
plt.show()

# The mathematical connections: centered data can be decomposed by SVD or by
# eigendecomposition of its covariance matrix. These small checks use NumPy.
X_centered = X - X.mean(axis=0)
_, singular_values, _ = np.linalg.svd(X_centered, full_matrices=False)
covariance = np.cov(X_centered, rowvar=False)
eigenvalues, _ = np.linalg.eigh(covariance)
print(f'Largest SVD variance value: {(singular_values[0] ** 2) / (len(X) - 1):.4f}')
print(f'Largest covariance eigenvalue: {eigenvalues[-1]:.4f}')

In [ ]:
# Visualize the top 10 Eigenfaces (Principal Components)
eigenfaces = pca.components_.reshape((n_components, faces.images.shape[1], faces.images.shape[2]))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(eigenfaces[i], cmap='gray')
    ax.set_title(f'Eigenface {i+1}')
    ax.axis('off')
plt.suptitle('Top 10 Eigenfaces (Principal Components)', fontsize=16)
plt.tight_layout()
plt.show()


## PCA, SVD, and eigendecomposition formulas

### Formula notation

- $X \in \mathbb{R}^{n\times d}$: real-valued image matrix with $n=400$ faces and $d=4096$ pixels per flattened image.
- $\boldsymbol{\mu}$: feature-wise mean image; $X_c$: centered data; $r$: retained number of components.
- $\mathbf{U}$, $\mathbf{S}$, and $\mathbf{V}$: matrices from Singular Value Decomposition; $\lambda_j$ and $\mathbf{v}_j$: eigenvalue and eigenvector $j$.

### Centering and covariance

$$X_c = X - \boldsymbol{\mu}$$

$$C = \frac{1}{n-1}X_c^T X_c$$

Centering is used so PCA describes variation around the average face, not variation caused by the overall pixel brightness. The covariance matrix $C$ measures which pixels vary together.

### Eigendecomposition and SVD

$$C\mathbf{v}_j = \lambda_j\mathbf{v}_j$$

$$X_c = \mathbf{U}\mathbf{S}\mathbf{V}^T$$

The principal directions are the eigenvectors of $C$, or equivalently the rows of $\mathbf{V}^T$ from SVD. Their eigenvalues equal $s_j^2/(n-1)$, where $s_j$ is singular value $j$. PCA keeps the $r$ directions with the largest eigenvalues because they explain the most data variation.

### Projection, reconstruction, and evaluation

$$Z = X_c\mathbf{V}_r$$

$$\hat{X} = Z\mathbf{V}_r^T + \boldsymbol{\mu}$$

$$\mathrm{ExplainedVariance}(r) = \frac{\sum_{j=1}^{r}\lambda_j}{\sum_{j=1}^{d}\lambda_j}$$

$$\mathrm{ReconstructionMSE} = \frac{1}{nd}\lVert X-\hat{X}\rVert_F^2$$

**Maximize** explained variance: it ranges from $0$ to $1$, and larger is better. **Minimize** reconstruction MSE: $0$ is perfect and there is no fixed worst value. PCA has no target labels, so these measures evaluate compression quality rather than classification accuracy.